**Note — notebooks découpés :** le même flux est réparti en **`depth_field_V3_2d.ipynb`** (segmentation, masques, profondeur Depth Pro, analyses 2D matplotlib, **export** `data/processed/depth_field_v3_bundle.npz`) et **`depth_field_V3_3d.ipynb`** (chargement du bundle, nuages Open3D, mesh Poisson, voxel / volume). Ce fichier reste une version **monolithique** de référence ; le script `src/_split_v3_notebooks.py` régénère les deux notebooks à partir de lui.

Notes expérience sel: poids initial: 33 grammes

dimensions mesurées 9x9x2cm

poids retitré: 8 grammes

hauteur caméra : 13 cm

## Structure du pipeline

1. **Decoupage + superposition des masques** — Segmentation SLIC/variance, rognage avec `pad`, redimensionnement sur la grille profondeur, puis combinaison (`mask_combined`).
2. **Generation des cartes de profondeur** — Inference depth_pro, normalisation et inversion (`revert_depth_image`) pour obtenir `depth_1` et `depth_2`.
3. **Adaptation du range de la deuxieme carte** — Construction de `depth_2_aligned`, puis recalage lineaire de son range pour le rendre comparable a `depth_1`.
4. **Generation 3D** — Masquage des cartes (`depth_1_masked`, `depth_2_masked`), differences, nuages de points et reliefs Open3D.

In [1]:
import sys
print("Python utilisé:", sys.executable)
print("Premiers chemins:", sys.path[:3])

import matplotlib
matplotlib.use('tkAgg')
from matplotlib import pyplot as plt
from PIL import Image
import torch
import numpy as np
import open3d as o3d
import requests
import dataclasses
from pathlib import Path
import cv2
import depth_pro
from depth_pro.depth_pro import create_model_and_transforms, DEFAULT_MONODEPTH_CONFIG_DICT
from transformers import GLPNImageProcessor, GLPNForDepthEstimation


Python utilisé: c:\Users\mvm\open3d_vision\.venv\Scripts\python.exe
Premiers chemins: ['C:\\Users\\mvm\\AppData\\Local\\Programs\\Python\\Python312\\python312.zip', 'C:\\Users\\mvm\\AppData\\Local\\Programs\\Python\\Python312\\DLLs', 'C:\\Users\\mvm\\AppData\\Local\\Programs\\Python\\Python312\\Lib']


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [67]:
def revert_depth_image(depth_image):
    """
    Inverse la profondeur de l'image : proche <-> lointain.
    depth_image : array numpy 2D (H, W), valeurs de profondeur.
    Retourne une copie avec depth_inv = depth_max - depth + depth_min (range préservé, ordre inversé).
    """
    d = np.asarray(depth_image, dtype=np.float64)
    d_min, d_max = d.min(), d.max()
    return (d_max - d + d_min).astype(depth_image.dtype if hasattr(depth_image, 'dtype') else np.float32)

Création des chemins des 4 images

In [68]:
# Génération des cartes de profondeur (ordre identique à V2, pour 4 images)
# 1) Chemins des images (équivalent IMAGE_PATH de V2)
DATA_DIR = Path(r"C:\Users\mvm\open3d_vision\data")
BASE_NAME_OLD = r"pile-of-soil-top-view-isolated-on-white-G1N4P8"
BASE_NAME_1 = r"C:\Users\mvm\open3d_vision\data\photos sel\33 grammes.jpg"
BASE_NAME_2 = r"C:\Users\mvm\open3d_vision\data\photos sel\25 grammes.jpg"
paths = [
    BASE_NAME_1,
    BASE_NAME_2,
]


### Étape 1 — Segmentation objet / fond (SLIC + variance)

Pipeline appliqué en parallèle aux deux images (BASE_NAME_1 et BASE_NAME_2) : image → CLAHE → SLIC → variance → masques mask_variance (image 1) et mask_variance_2 (image 2).  
Ce masque sert ensuite à isoler l’objet dans les cartes de profondeur.

In [108]:
# Pipeline top-hat + Otsu + SLIC pour les deux images (paths[0] et paths[1]) en parallèle
import cv2 as cv
from skimage.segmentation import slic, mark_boundaries

def run_slic_pipeline(path, kernel, clahe):
    """Applique CLAHE, top-hat, Otsu et SLIC sur une image. Retourne un dict avec toutes les sorties."""
    img_th = cv.imread(str(path), cv.IMREAD_GRAYSCALE)
    assert img_th is not None, f"Image non trouvée: {path}"
    equalized = cv.equalizeHist(img_th)
    clahe_eq = clahe.apply(img_th)
    tophat = cv.morphologyEx(equalized, cv.MORPH_TOPHAT, kernel)
    _, th_otsu = cv.threshold(clahe_eq, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
    img_rgb = cv.cvtColor(clahe_eq, cv.COLOR_GRAY2BGR)
    segments = slic(img_rgb, n_segments=250, compactness=10, sigma=1, start_label=1)
    img_slic = (np.clip(mark_boundaries(img_rgb, segments, color=(1, 0, 0), mode="thick"), 0, 1) * 255).astype(np.uint8)
    return {"img_th": img_th, "equalized": equalized, "clahe_equalized": clahe_eq, "tophat": tophat,
            "th_otsu": th_otsu, "img_rgb": img_rgb, "segments": segments, "img_slic": img_slic}

kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (25, 25))
clahe = cv.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

# Traitement des deux images (BASE_NAME_1 et BASE_NAME_2)
out_1 = run_slic_pipeline(paths[0], kernel, clahe)
out_2 = run_slic_pipeline(paths[1], kernel, clahe)

# Variables pour la suite : image 1 (compatibilité avec cellule variance)
img_th = out_1["img_th"]
equalized = out_1["equalized"]
clahe_equalized = out_1["clahe_equalized"]
tophat = out_1["tophat"]
th_tophat_otsu = out_1["th_otsu"]
img_rgb = out_1["img_rgb"]
segments = out_1["segments"]
img_slic = out_1["img_slic"]
# Image 2
img_th_2 = out_2["img_th"]
segments_2 = out_2["segments"]

# Visualisation : 2 lignes = image 1, image 2 ; 3 colonnes = Original, CLAHE+Otsu, SLIC
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for i, (out, title) in enumerate([(out_1, "Image 1 (33 g)"), (out_2, "Image 2 (25 g)")]):
    axes[i, 0].imshow(out["img_th"], cmap="gray")
    axes[i, 0].set_title(f"{title} — Original")
    axes[i, 0].axis("off")
    axes[i, 1].imshow(out["th_otsu"], cmap="gray")
    axes[i, 1].set_title(f"{title} — Otsu")
    axes[i, 1].axis("off")
    axes[i, 2].imshow(cv.cvtColor(out["img_slic"], cv.COLOR_BGR2RGB))
    axes[i, 2].set_title(f"{title} — SLIC superpixels")
    axes[i, 2].axis("off")
plt.tight_layout()
plt.show()

In [109]:
# Séparation binaire par variance des superpixels pour les deux images
def mask_from_variance(img_grayscale, segments, percentile=68):
    """Masque binaire : 255 = superpixels texturés (variance ≥ percentile), 0 = reste."""
    gray = np.asarray(img_grayscale, dtype=np.float64)
    labels = np.unique(segments)
    var_per_label = {lab: np.var(gray[segments == lab]) for lab in labels}
    vars_arr = np.array([var_per_label[lab] for lab in labels])
    thresh = np.percentile(vars_arr, percentile)
    interesting = set(lab for lab, v in var_per_label.items() if v >= thresh)
    return np.where(np.isin(segments, list(interesting)), 255, 0).astype(np.uint8)

percentile_var = 68  # paramètre commun aux deux images
mask_variance = mask_from_variance(img_th, segments, percentile_var)
mask_variance_2 = mask_from_variance(img_th_2, segments_2, percentile_var)

# Visualisation : 2 lignes (image 1, image 2) — original, masque, superposition
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for i, (im, mask, title) in enumerate([
    (img_th, mask_variance, "Image 1 (33 g)"),
    (img_th_2, mask_variance_2, "Image 2 (25 g)"),
]):
    axes[i, 0].imshow(im, cmap="gray")
    axes[i, 0].set_title(f"{title} — Original")
    axes[i, 0].axis("off")
    axes[i, 1].imshow(mask, cmap="gray")
    axes[i, 1].set_title(f"{title} — Masque variance (p{percentile_var})")
    axes[i, 1].axis("off")
    overlay = cv.cvtColor(im, cv.COLOR_GRAY2BGR)
    overlay[mask == 255] = [0, 180, 0]
    axes[i, 2].imshow(cv.cvtColor(overlay, cv.COLOR_BGR2RGB))
    axes[i, 2].set_title(f"{title} — Superpixels retenus")
    axes[i, 2].axis("off")
plt.tight_layout()
plt.show()

### Etape 2 — Generation des cartes de profondeur

Creation des objets image et initialisation du modele depth-pro.

In [110]:
# 2) Modèle depth_pro + chargement / transform / overwrite PIL (ordre V2 strict)
CHECKPOINT = Path(r"C:\Users\mvm\open3d_vision\ml-depth-pro\checkpoints\depth_pro_alt.pt")
config = dataclasses.replace(DEFAULT_MONODEPTH_CONFIG_DICT, checkpoint_uri=str(CHECKPOINT))
model, transform = create_model_and_transforms(config=config)
model.eval()

# Image 1 : load_rgb → transform → overwrite avec PIL (comme V2)
image_og_1, _, f_px_1 = depth_pro.load_rgb(str(paths[0]))
image_1 = transform(image_og_1)
image_1 = Image.open(paths[0]).convert("RGB")
# Image 2
image_og_2, _, f_px_2 = depth_pro.load_rgb(str(paths[1]))
image_2 = transform(image_og_2)
image_2 = Image.open(paths[1]).convert("RGB")
# # Image 3
# image_og_3, _, f_px_3 = depth_pro.load_rgb(str(paths[2]))
# image_3 = transform(image_og_3)
# image_3 = Image.open(paths[2]).convert("RGB")
# # Image 4
# image_og_4, _, f_px_4 = depth_pro.load_rgb(str(paths[3]))
# image_4 = transform(image_og_4)
# image_4 = Image.open(paths[3]).convert("RGB")


Inférence des images pour cartes de profondeur

In [72]:
# 3) Inférence depth_pro pour chaque image (ordre V2)
# On doit passer à model.infer le tenseur transformé, pas l'image PIL !
prediction_1 = model.infer(transform(image_og_1), f_px=f_px_1)
depth_1 = prediction_1["depth"].squeeze().cpu().numpy()
prediction_2 = model.infer(transform(image_og_2), f_px=f_px_2)
depth_2 = prediction_2["depth"].squeeze().cpu().numpy()
# prediction_3 = model.infer(transform(image_og_3), f_px=f_px_3)
# depth_3 = prediction_3["depth"].squeeze().cpu().numpy()
# prediction_4 = model.infer(transform(image_og_4), f_px=f_px_4)
# depth_4 = prediction_4["depth"].squeeze().cpu().numpy()


In [111]:
# Normalisation des cartes de profondeur (min=0, max=1)
def normalize_depth_map(depth):
    d_min = depth.min()
    d_max = depth.max()
    if d_max > d_min:
        return (depth - d_min) / (d_max - d_min)
    else:
        return np.zeros_like(depth)

depth_1 = normalize_depth_map(depth_1)
depth_2 = normalize_depth_map(depth_2)
# depth_3, depth_4 : désactivés (images 3 et 4 non chargées)
# depth_3 = normalize_depth_map(depth_3)
# depth_4 = normalize_depth_map(depth_4)

In [112]:
# 5) Rognage pad 
pad = 16
image_cropped_1 = image_1.crop((pad, pad, image_1.width - pad, image_1.height - pad))
image_cropped_2 = image_2.crop((pad, pad, image_2.width - pad, image_2.height - pad))
# image_cropped_3 = image_3.crop((pad, pad, image_3.width - pad, image_3.height - pad))
# image_cropped_4 = image_4.crop((pad, pad, image_4.width - pad, image_4.height - pad))


In [124]:
# 6) Revert depth 
depth_1 = revert_depth_image(depth_1)
depth_2 = revert_depth_image(depth_2)
# depth_3, depth_4 : désactivés
# depth_3 = revert_depth_image(depth_3)
# depth_4 = revert_depth_image(depth_4)

### Etape 1 — Decoupage + superposition des masques

Les masques SLIC (`mask_variance`, `mask_variance_2`) sont rognees avec le meme `pad` que les images, redimensionnes sur la grille profondeur, puis combines (`mask_combined`).
Cette etape fixe la zone objet utilisee ensuite pour l'adaptation de range et la 3D.

In [125]:
# Decoupage + superposition des masques sur la grille profondeur de l'image 1
pad = 16  # meme valeur que le rognage des images RGB
h_d1, w_d1 = depth_1.shape

def crop_resize_mask(mask_var, pad, h_d, w_d):
    """Rogne le masque avec pad puis le redimensionne sur (h_d, w_d)."""
    if mask_var.shape[0] > 2 * pad and mask_var.shape[1] > 2 * pad:
        mask_crop = mask_var[pad:-pad, pad:-pad]
    else:
        mask_crop = mask_var
    return cv2.resize(mask_crop, (w_d, h_d), interpolation=cv2.INTER_NEAREST)

# Base commune: la 2e depth map est alignee sur la grille de depth_1
if depth_2.shape != (h_d1, w_d1):
    depth_2_aligned = cv2.resize(
        depth_2.astype(np.float32), (w_d1, h_d1), interpolation=cv2.INTER_LINEAR
    ).astype(np.float64)
else:
    depth_2_aligned = np.asarray(depth_2, dtype=np.float64)

# Masques objets alignes sur la meme grille
mask_obj = crop_resize_mask(mask_variance, pad, h_d1, w_d1)
mask_obj_2 = crop_resize_mask(mask_variance_2, pad, h_d1, w_d1)
mask_combined = np.maximum(mask_obj, mask_obj_2)

# Visualisation du decoupage et de la superposition
rgb1_show = np.asarray(image_cropped_1.convert("RGB"), dtype=np.uint8)
if rgb1_show.shape[:2] != (h_d1, w_d1):
    rgb1_show = np.asarray(
        Image.fromarray(rgb1_show).resize((w_d1, h_d1), Image.Resampling.LANCZOS),
        dtype=np.uint8,
    )

overlay_1 = rgb1_show.copy()
overlay_1[mask_obj == 255] = [255, 70, 70]
overlay_2 = rgb1_show.copy()
overlay_2[mask_obj_2 == 255] = [70, 255, 70]
overlay_union = rgb1_show.copy()
overlay_union[mask_combined == 255] = [255, 220, 0]

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes[0, 0].imshow(mask_obj, cmap="gray")
axes[0, 0].set_title("mask_obj (image 1 aligne)")
axes[0, 0].axis("off")
axes[0, 1].imshow(mask_obj_2, cmap="gray")
axes[0, 1].set_title("mask_obj_2 (image 2 aligne)")
axes[0, 1].axis("off")
axes[0, 2].imshow(mask_combined, cmap="gray")
axes[0, 2].set_title("mask_combined (union)")
axes[0, 2].axis("off")

axes[1, 0].imshow(overlay_1)
axes[1, 0].set_title("Superposition mask_obj")
axes[1, 0].axis("off")
axes[1, 1].imshow(overlay_2)
axes[1, 1].set_title("Superposition mask_obj_2")
axes[1, 1].axis("off")
axes[1, 2].imshow(overlay_union)
axes[1, 2].set_title("Superposition union")
axes[1, 2].axis("off")

plt.tight_layout()
plt.show()

In [127]:
# (Fin de la procédure cartes de profondeur — suite 3D éventuelle ci-dessous)
fig, ax = plt.subplots(2, 2, figsize=(8, 14))
for i, (img_crop, d) in enumerate([
    (image_cropped_1, depth_1),
    (image_cropped_2, depth_2),
    # (image_cropped_3, depth_3),
    # (image_cropped_4, depth_4),
]):
    ax[i, 0].imshow(img_crop)
    ax[i, 0].set_title(f"Image rognée {i + 1}")
    ax[i, 0].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    ax[i, 1].imshow(d, cmap="plasma")
    ax[i, 1].set_title(f"Profondeur GLPN {i + 1}")
    ax[i, 1].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
plt.tight_layout()
plt.show()

In [128]:
# Comparaison des deux images : RGB et profondeur (plus de flip / soustraction)
fig, ax = plt.subplots(2, 2, figsize=(12, 10))
ax[0, 0].imshow(image_cropped_1)
ax[0, 0].set_title("Image 1 (33 g) — RGB")
ax[0, 0].axis("off")
ax[0, 1].imshow(image_cropped_2)
ax[0, 1].set_title("Image 2 (25 g) — RGB")
ax[0, 1].axis("off")
im0 = ax[1, 0].imshow(depth_1, cmap='plasma')
ax[1, 0].set_title("Image 1 — Profondeur")
ax[1, 0].axis("off")
plt.colorbar(im0, ax=ax[1, 0], fraction=0.046)
im1 = ax[1, 1].imshow(depth_2, cmap='plasma')
ax[1, 1].set_title("Image 2 — Profondeur")
ax[1, 1].axis("off")
plt.colorbar(im1, ax=ax[1, 1], fraction=0.046)
plt.tight_layout()
plt.show()


In [154]:
def mask_rgb_strong_similarity(rgb_1, rgb_2, percentile_similar=8, morph_open_ksize=3):
    """
    Masque binaire : 255 = pixels où les deux images RGB sont très similaires (faible ΔE Lab).
    Les images sont redimensionnées pour correspondre à la première si les tailles diffèrent.
    percentile_similar : percentile bas sur la carte ΔE (plus bas = critère plus strict).
    Retourne (masque uint8, carte delta_e, seuil utilisé).
    """
    a = np.asarray(rgb_1.convert("RGB") if hasattr(rgb_1, "convert") else rgb_1, dtype=np.uint8)
    b = np.asarray(rgb_2.convert("RGB") if hasattr(rgb_2, "convert") else rgb_2, dtype=np.uint8)
    if a.shape[:2] != b.shape[:2]:
        b = np.asarray(
            Image.fromarray(b).resize((a.shape[1], a.shape[0]), Image.Resampling.LANCZOS),
            dtype=np.uint8,
        )
    lab1 = cv2.cvtColor(a, cv2.COLOR_RGB2LAB).astype(np.float64)
    lab2 = cv2.cvtColor(b, cv2.COLOR_RGB2LAB).astype(np.float64)
    delta_e = np.sqrt(np.sum((lab1 - lab2) ** 2, axis=2))
    thresh = np.percentile(delta_e, percentile_similar)
    mask = ((delta_e <= thresh).astype(np.uint8)) * 255
    if morph_open_ksize and morph_open_ksize > 0:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (morph_open_ksize, morph_open_ksize))
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k)
    return mask, delta_e, float(thresh)


def delta_e_masked_by_common(delta_e, mask_common_uint8):
    """
    Retourne la carte ΔE avec NaN hors du masque commun (255 = zone conservée).
    """
    m = mask_common_uint8.astype(np.uint8)
    return np.where(m == 255, delta_e, np.nan)


# RGB rognées comme le reste du pipeline (même pad que depth)
P_SIM_RGB = 8  # abaisser pour masque plus strict (moins de pixels)

mask_rgb_similar_raw, delta_e_map, thresh_de = mask_rgb_strong_similarity(
    image_cropped_1, image_cropped_2, percentile_similar=P_SIM_RGB
)

# Aligner mask_combined (taille depth) sur les RGB rognées : similarité uniquement DANS le masque commun
h_sim, w_sim = mask_rgb_similar_raw.shape[:2]
mask_combined_for_rgb = cv2.resize(
    mask_combined, (w_sim, h_sim), interpolation=cv2.INTER_NEAREST
)
delta_e_in_common = delta_e_masked_by_common(delta_e_map, mask_combined_for_rgb)
# Zones très similaires (binaire) ∩ objet SLIC commun
mask_rgb_similar = delta_e_masked_by_common(
    mask_rgb_similar_raw, mask_combined_for_rgb
)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes[0, 0].imshow(image_cropped_1)
axes[0, 0].set_title("Image 1 — RGB")
axes[0, 0].axis("off")
axes[0, 1].imshow(image_cropped_2)
axes[0, 1].set_title("Image 2 — RGB")
axes[0, 1].axis("off")
im_de = axes[0, 2].imshow(delta_e_in_common, cmap="magma")
axes[0, 2].set_title("ΔE (Lab) — uniquement dans mask_combined (↓ = plus similaire)")
axes[0, 2].axis("off")
plt.colorbar(im_de, ax=axes[0, 2], fraction=0.046, label="ΔE")
axes[1, 0].imshow(mask_combined_for_rgb, cmap="gray")
axes[1, 0].set_title("mask_combined (redim. → RGB)")
axes[1, 0].axis("off")
axes[1, 1].imshow(mask_rgb_similar_raw, cmap="gray")
axes[1, 1].set_title(f"Très similaires (ΔE ≤ p{P_SIM_RGB} ≈ {thresh_de:.1f}), plein cadre")
axes[1, 1].axis("off")
axes[1, 2].imshow(mask_rgb_similar, cmap="gray")
axes[1, 2].set_title("Très similaires ∩ mask_combined")
axes[1, 2].axis("off")
plt.tight_layout()
plt.show()

In [155]:
def pixels_coords_mask_255(mask_uint8):
    """
    Retourne les coordonnées de tous les pixels où le masque vaut 255.
    - coords_row_col : (row, col), indices numpy / imshow
    - coords_xy : (x, y) avec x = colonne, y = ligne
    """
    row, col = np.where(np.asarray(mask_uint8) == 255)
    coords_row_col = np.column_stack((row, col))
    coords_xy = np.column_stack((col, row))
    return coords_row_col, coords_xy


coords_row_col, coords_xy = pixels_coords_mask_255(mask_rgb_similar)
print(mask_rgb_similar.shape)
print(f"Nombre de pixels à 255 : {len(coords_row_col)}")
print(f"Exemple (5 premiers) row,col : {coords_row_col[:5]}")
print(f"Exemple (5 premiers) x,y   : {coords_xy[:5]}")

(1811, 1350)
Nombre de pixels à 255 : 10331
Exemple (5 premiers) row,col : [[272 673]
 [272 674]
 [272 675]
 [272 676]
 [272 677]]
Exemple (5 premiers) x,y   : [[673 272]
 [674 272]
 [675 272]
 [676 272]
 [677 272]]


In [156]:
# Moyenne de (depth_1 - depth_2) sur les pixels de coords_xy
# Convention : coords_xy = (x, y) avec x = colonne, y = ligne → indexation depth[y, x]
h1, w1 = depth_1.shape
if depth_2.shape != (h1, w1):
    depth_2_aligned = cv2.resize(
        depth_2.astype(np.float32), (w1, h1), interpolation=cv2.INTER_LINEAR
    )
else:
    depth_2_aligned = np.asarray(depth_2, dtype=np.float64)

x_coords = coords_xy[:, 0]
y_coords = coords_xy[:, 1]
in_bounds = (
    (y_coords >= 0) & (y_coords < h1) & (x_coords >= 0) & (x_coords < w1)
)
if not np.all(in_bounds):
    print(f"Avertissement : {int(np.sum(~in_bounds))} coordonnée(s) hors limites — ignorées.")
x_coords = x_coords[in_bounds]
y_coords = y_coords[in_bounds]

d1_at_pts = depth_1[y_coords, x_coords].astype(np.float64)
d2_at_pts = depth_2_aligned[y_coords, x_coords].astype(np.float64)
mean_diff_depth_1_minus_2 = float(np.mean(d1_at_pts - d2_at_pts))
print(f"Moyenne (depth_1 - depth_2) sur coords_xy : {mean_diff_depth_1_minus_2}")


Moyenne (depth_1 - depth_2) sur coords_xy : -0.10643231654118712


In [157]:
# Etape 3 — Adaptation du range de depth_2_aligned pour comparaison avec depth_1
# On applique un recalage lineaire de range sur la zone utile (mask_combined).

d1 = np.asarray(depth_1, dtype=np.float64)
d2a = np.asarray(depth_2_aligned, dtype=np.float64)

if d1.shape != d2a.shape:
    raise ValueError("depth_1 et depth_2_aligned doivent avoir la meme forme avant remap.")

if "mask_combined" not in locals():
    raise ValueError("mask_combined manquant: execute l'etape 1 (cellule masques) avant cette cellule.")

valid = (mask_combined == 255) & np.isfinite(d1) & np.isfinite(d2a)
if not np.any(valid):
    raise ValueError("Aucun pixel valide pour adapter le range de depth_2_aligned.")

# Ranges robustes (percentiles) pour limiter l'effet des outliers.
d1_lo, d1_hi = np.percentile(d1[valid], [1, 99])
d2_lo, d2_hi = np.percentile(d2a[valid], [1, 99])

if (d2_hi - d2_lo) <= 1e-12:
    depth_2_aligned = np.full_like(d2a, float(d1_lo))
else:
    depth_2_aligned = (d2a - d2_lo) / (d2_hi - d2_lo)
    depth_2_aligned = depth_2_aligned * (d1_hi - d1_lo) + d1_lo

print(f"Range depth_1 (zone valide): [{d1_lo:.6f}, {d1_hi:.6f}]")
print(f"Range depth_2_aligned avant remap: [{d2_lo:.6f}, {d2_hi:.6f}]")
print(
    "Range depth_2_aligned apres remap (global): "
    f"[{float(np.nanmin(depth_2_aligned)):.6f}, {float(np.nanmax(depth_2_aligned)):.6f}]"
)


Range depth_1 (zone valide): [0.436376, 0.965917]
Range depth_2_aligned avant remap: [0.497728, 0.971034]
Range depth_2_aligned apres remap (global): [-0.120488, 0.998325]


In [158]:
# depth_1, depth_2_aligned et RGB associées (même grille que les cartes de profondeur)
h_d, w_d = depth_1.shape
rgb1 = np.asarray(image_cropped_1)
if rgb1.dtype != np.uint8:
    rgb1 = (np.clip(rgb1, 0, 1) * 255).astype(np.uint8)
if rgb1.shape[0] != h_d or rgb1.shape[1] != w_d:
    rgb1 = np.asarray(
        Image.fromarray(rgb1).resize((w_d, h_d), Image.Resampling.LANCZOS),
        dtype=np.uint8,
    )

rgb2 = np.asarray(image_cropped_2)
if rgb2.dtype != np.uint8:
    rgb2 = (np.clip(rgb2, 0, 1) * 255).astype(np.uint8)
if rgb2.shape[0] != h_d or rgb2.shape[1] != w_d:
    rgb2 = np.asarray(
        Image.fromarray(rgb2).resize((w_d, h_d), Image.Resampling.LANCZOS),
        dtype=np.uint8,
    )

d2_show = np.asarray(depth_2_aligned, dtype=np.float64)
if d2_show.shape != (h_d, w_d):
    d2_show = cv2.resize(
        d2_show.astype(np.float32), (w_d, h_d), interpolation=cv2.INTER_LINEAR
    )

# Amélioration de la disposition et de la taille des images
# Nouvelle disposition : 2x3 grille, agrandissement des images RGB et profondeur side-by-side, puis la différence sur toute la largeur

fig = plt.figure(figsize=(18, 12))  # Taille augmentée

# Grille 2x3: [RGB1 | depth1 | filler][RGB2 | depth2_aligned | filler], puis la différence en dessous sur 1x3
from matplotlib import gridspec
gs = gridspec.GridSpec(3, 3, height_ratios=[1.2, 1.2, 1.2], hspace=0.30, wspace=0.20)

# Premières lignes : chaque image RGB à gauche, profondeur à droite
ax00 = fig.add_subplot(gs[0, 0])  # RGB1
ax01 = fig.add_subplot(gs[0, 1])  # depth1
ax10 = fig.add_subplot(gs[1, 0])  # RGB2
ax11 = fig.add_subplot(gs[1, 1])  # depth2_aligned

# Étalement sur 2 colonnes et 3ème ligne complète pour la différence
ax_diff = fig.add_subplot(gs[2, :2])  # Différence sur deux colonnes

# Si on veut un panneau libre pour d'autres visus ou du texte :
# ax_filler1 = fig.add_subplot(gs[0, 2])
# ax_filler2 = fig.add_subplot(gs[1, 2])
# for ax in [ax_filler1, ax_filler2]: ax.axis("off")

# Calcul des bornes pour depth2_aligned
d2_lo, d2_hi = float(np.min(d2_show)), float(np.max(d2_show))
d1_f = np.asarray(depth_1, dtype=np.float64)
diff_depth = d1_f - d2_show
abs_max = float(np.max(np.abs(diff_depth))) if diff_depth.size else 0.0
if abs_max < 1e-12:
    abs_max = 1.0

# Affichages
ax00.imshow(rgb1)
ax00.set_title("Image 1 (33 g) — RGB", fontsize=14)
ax00.axis("off")

im0 = ax01.imshow(depth_1, cmap="plasma")
ax01.set_title("depth_1", fontsize=14)
ax01.axis("off")
cbar0 = fig.colorbar(im0, ax=ax01, fraction=0.046)
cbar0.ax.tick_params(labelsize=10)

ax10.imshow(rgb2)
ax10.set_title("Image 2 (25 g) — RGB", fontsize=14)
ax10.axis("off")

im1 = ax11.imshow(d2_show, cmap="plasma", vmin=0, vmax=1)
ax11.set_title(
    f"depth_2_aligned — échelle couleur [0, 1]\nvaleurs [{d2_lo:.4g}, {d2_hi:.4g}]", fontsize=14
)
ax11.axis("off")
cbar1 = fig.colorbar(im1, ax=ax11, fraction=0.046, label="intensité colormap (0–1)")
cbar1.ax.tick_params(labelsize=10)

im_d = ax_diff.imshow(diff_depth, cmap="coolwarm", vmin=-abs_max, vmax=abs_max)
ax_diff.set_title(
    f"depth_1 − depth_2_aligned (coolwarm, ±{abs_max:.4g}) — "
    f"moyenne={float(np.mean(diff_depth)):.4g}, écart-type={float(np.std(diff_depth)):.4g}",
    fontsize=14
)
ax_diff.axis("off")
cbar_d = fig.colorbar(im_d, ax=ax_diff, fraction=0.08, pad=0.02, label="Δ profondeur")
cbar_d.ax.tick_params(labelsize=10)

plt.tight_layout()
plt.show()

C:\Users\mvm\AppData\Local\Temp\ipykernel_14596\159736704.py:91: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


In [159]:
# Etape 4 — Preparation des cartes masquées pour la 3D
# Cette cellule suppose que depth_2_aligned a deja ete calculee puis remappee (etape 3).
if "depth_2_aligned" not in locals():
    raise ValueError("depth_2_aligned manquant: execute l'etape 3 avant la 3D.")

h_d1, w_d1 = depth_1.shape
depth_2_aligned = np.asarray(depth_2_aligned, dtype=np.float64)
if depth_2_aligned.shape != (h_d1, w_d1):
    depth_2_aligned = cv2.resize(
        depth_2_aligned.astype(np.float32), (w_d1, h_d1), interpolation=cv2.INTER_LINEAR
    ).astype(np.float64)

depth_1_masked = np.where(mask_combined == 255, depth_1.astype(np.float64), np.nan)
depth_2_masked = np.where(mask_combined == 255, depth_2_aligned, np.nan)

# Normalisation pour l'affichage uniquement
d1_min, d1_max = np.nanmin(depth_1), np.nanmax(depth_1)
depth_1_norm = (depth_1 - d1_min) / (d1_max - d1_min + 1e-8) if d1_max > d1_min else np.zeros_like(depth_1)
d2_min, d2_max = np.nanmin(depth_2_aligned), np.nanmax(depth_2_aligned)
depth_2_norm = (
    (depth_2_aligned - d2_min) / (d2_max - d2_min + 1e-8)
    if d2_max > d2_min else np.zeros_like(depth_2_aligned)
)

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes[0, 0].imshow(depth_1_norm, cmap="plasma")
axes[0, 0].set_title("Image 1 — depth_1 (norm affichee)")
axes[0, 0].axis("off")
axes[0, 1].imshow(mask_combined, cmap="gray")
axes[0, 1].set_title("Masque commun (union)")
axes[0, 1].axis("off")
axes[0, 2].imshow(depth_1_masked, cmap="plasma")
axes[0, 2].set_title("Image 1 — depth_1 masquee")
axes[0, 2].axis("off")

axes[1, 0].imshow(depth_2_norm, cmap="plasma")
axes[1, 0].set_title("Image 2 — depth_2_aligned (norm affichee)")
axes[1, 0].axis("off")
axes[1, 1].imshow(mask_obj_2, cmap="gray")
axes[1, 1].set_title("Image 2 — masque aligne")
axes[1, 1].axis("off")
axes[1, 2].imshow(depth_2_masked, cmap="plasma")
axes[1, 2].set_title("Image 2 — depth_2_aligned masquee")
axes[1, 2].axis("off")

plt.tight_layout()
plt.show()

In [160]:
# depth_1, depth_2_aligned et RGB associées (même grille que les cartes de profondeur)
h_d, w_d = depth_1.shape
rgb1 = np.asarray(image_cropped_1)
if rgb1.dtype != np.uint8:
    rgb1 = (np.clip(rgb1, 0, 1) * 255).astype(np.uint8)
if rgb1.shape[0] != h_d or rgb1.shape[1] != w_d:
    rgb1 = np.asarray(
        Image.fromarray(rgb1).resize((w_d, h_d), Image.Resampling.LANCZOS),
        dtype=np.uint8,
    )

rgb2 = np.asarray(image_cropped_2)
if rgb2.dtype != np.uint8:
    rgb2 = (np.clip(rgb2, 0, 1) * 255).astype(np.uint8)
if rgb2.shape[0] != h_d or rgb2.shape[1] != w_d:
    rgb2 = np.asarray(
        Image.fromarray(rgb2).resize((w_d, h_d), Image.Resampling.LANCZOS),
        dtype=np.uint8,
    )

d2_show = np.asarray(depth_2_aligned, dtype=np.float64)
if d2_show.shape != (h_d, w_d):
    d2_show = cv2.resize(
        d2_show.astype(np.float32), (w_d, h_d), interpolation=cv2.INTER_LINEAR
    )

# Appliquer le masque commun
depth_1_masked_for_show = np.where(mask_combined == 255, depth_1.astype(np.float64), np.nan)
d2_show_masked = np.where(mask_combined == 255, d2_show, np.nan)

# Amélioration de la disposition et de la taille des images
# Nouvelle disposition : 2x3 grille, agrandissement des images RGB et profondeur side-by-side, puis la différence sur toute la largeur

fig = plt.figure(figsize=(18, 12))  # Taille augmentée

# Grille 2x3: [RGB1 | depth1 | filler][RGB2 | depth2_aligned | filler], puis la différence en dessous sur 1x3
from matplotlib import gridspec
gs = gridspec.GridSpec(3, 3, height_ratios=[1.2, 1.2, 1.2], hspace=0.30, wspace=0.20)

# Premières lignes : chaque image RGB à gauche, profondeur à droite
ax00 = fig.add_subplot(gs[0, 0])  # RGB1
ax01 = fig.add_subplot(gs[0, 1])  # depth1 masked
ax10 = fig.add_subplot(gs[1, 0])  # RGB2
ax11 = fig.add_subplot(gs[1, 1])  # depth2_aligned masked

# Étalement sur 2 colonnes et 3ème ligne complète pour la différence
ax_diff = fig.add_subplot(gs[2, :2])  # Différence sur deux colonnes

# Calcul des bornes pour depth2_aligned masked
finite_mask = np.isfinite(d2_show_masked)
if np.any(finite_mask):
    d2_lo, d2_hi = float(np.nanmin(d2_show_masked)), float(np.nanmax(d2_show_masked))
else:
    d2_lo, d2_hi = 0.0, 1.0
d1_f_masked = np.asarray(depth_1_masked_for_show, dtype=np.float64)
diff_depth_masked = d1_f_masked - d2_show_masked
abs_max = float(np.nanmax(np.abs(diff_depth_masked))) if np.any(np.isfinite(diff_depth_masked)) else 1.0
if abs_max < 1e-12:
    abs_max = 1.0

# Affichages
ax00.imshow(rgb1)
ax00.set_title("Image 1 (33 g) — RGB", fontsize=14)
ax00.axis("off")

im0 = ax01.imshow(depth_1_masked_for_show, cmap="plasma")
ax01.set_title("depth_1 (après masque commun)", fontsize=14)
ax01.axis("off")
cbar0 = fig.colorbar(im0, ax=ax01, fraction=0.046)
cbar0.ax.tick_params(labelsize=10)

ax10.imshow(rgb2)
ax10.set_title("Image 2 (25 g) — RGB", fontsize=14)
ax10.axis("off")

im1 = ax11.imshow(d2_show_masked, cmap="plasma", vmin=0, vmax=1)
ax11.set_title(
    f"depth_2_aligned (après masque commun)\n[0, 1], valeurs [{d2_lo:.4g}, {d2_hi:.4g}]", fontsize=14
)
ax11.axis("off")
cbar1 = fig.colorbar(im1, ax=ax11, fraction=0.046, label="intensité colormap (0–1)")
cbar1.ax.tick_params(labelsize=10)

im_d = ax_diff.imshow(diff_depth_masked, cmap="coolwarm", vmin=-abs_max, vmax=abs_max)
mean_diff = float(np.nanmean(diff_depth_masked)) if np.any(np.isfinite(diff_depth_masked)) else 0.0
std_diff = float(np.nanstd(diff_depth_masked)) if np.any(np.isfinite(diff_depth_masked)) else 0.0
ax_diff.set_title(
    f"moyenne={mean_diff:.4g}, écart-type={std_diff:.4g}",
    fontsize=14
)
ax_diff.axis("off")
cbar_d = fig.colorbar(im_d, ax=ax_diff, fraction=0.08, pad=0.02, label="Δ profondeur")
cbar_d.ax.tick_params(labelsize=10)

plt.tight_layout()
plt.show()

C:\Users\mvm\AppData\Local\Temp\ipykernel_14596\774362203.py:95: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


### Etape 4 — Generation 3D (preparation)

Les cartes de profondeur sont d'abord masquees avec `mask_combined`.
Pour l'image 2, la source Z est **`depth_2_aligned`** (etape 3), puis `depth_2_masked` alimente toutes les representations 3D et les differences.

**Nuages deux images :** XY normalisés par `max(h,w)`, **même position** (superposition), **Z = valeurs brutes** des `depth_map` masquées (sans remise à l’échelle), teintes rouge (img1) / bleue (img2). Axe Z : ligne bleue + graduations sur la plage de profondeur affichée.

In [161]:
# Nuage de points Image 1 : plan XY normalisé par max(h,w), Z = valeurs brutes de depth_1_masked (même échelle que la depth map)
import open3d as o3d

rgb_img = np.array(image_cropped_1)
if rgb_img.dtype != np.uint8:
    rgb_img = (np.clip(rgb_img, 0, 1) * 255).astype(np.uint8)
h, w = depth_1_masked.shape
if rgb_img.shape[0] != h or rgb_img.shape[1] != w:
    rgb_img = np.asarray(Image.fromarray(rgb_img).resize((w, h), Image.Resampling.LANCZOS), dtype=np.uint8)
xx, yy = np.meshgrid(np.arange(w), np.arange(h), indexing="xy")

zv = depth_1_masked[np.isfinite(depth_1_masked)]
if zv.size == 0:
    raise ValueError("depth_1_masked : aucun pixel valide pour le nuage de points.")
z_min, z_max = float(zv.min()), float(zv.max())
z_span = max(z_max - z_min, 1e-12)
z_std = float(np.std(zv))
n_unique = len(np.unique(np.round(zv.astype(np.float64), 8)))
print(
    f"Img1 — Z brut (depth_map): min={z_min:.6g}, max={z_max:.6g}, span={z_span:.6g}, "
    f"std={z_std:.6g}, valeurs uniques (~8 déc.)={n_unique}"
)
if z_span < 1e-9:
    raise ValueError("Toutes les profondeurs Z sont identiques (span ~ 0) — vérifier depth_1 / masque.")

s_xy = float(max(w, h))
x_n = xx.astype(np.float64) / s_xy
y_n = yy.astype(np.float64) / s_xy
z_n = np.where(
    np.isfinite(depth_1_masked),
    depth_1_masked.astype(np.float64),
    np.nan,
)
points = np.stack([x_n, y_n, z_n], axis=-1).reshape(-1, 3)
tint_1 = np.array([0.92, 0.38, 0.28])
base_rgb = rgb_img.reshape(-1, 3).astype(np.float64) / 255.0
colors = np.clip(0.5 * base_rgb + 0.5 * tint_1, 0.0, 1.0)
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)
pcd.colors = o3d.utility.Vector3dVector(colors)
valid = np.isfinite(points[:, 2])
pcd = pcd.select_by_index(np.where(valid)[0])
z_vis = np.asarray(pcd.points)[:, 2]
print(
    f"Img1 — Z nuage (= depth_map): min={z_vis.min():.6g}, max={z_vis.max():.6g}, "
    f"span={float(z_vis.max() - z_vis.min()):.6g}"
)

cx, cy = 0.5 * (w - 1) / s_xy, 0.5 * (h - 1) / s_xy
axis_size = max(0.08, 0.15 * z_span)
frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=axis_size, origin=[cx, cy, z_min])

# Axe Z (bleu) sur la plage réelle des profondeurs [z_min, z_max]
line_z_pts = np.array([[cx, cy, z_min], [cx, cy, z_max]], dtype=np.float64)
line_z = o3d.geometry.LineSet()
line_z.points = o3d.utility.Vector3dVector(line_z_pts)
line_z.lines = o3d.utility.Vector2iVector(np.array([[0, 1]], dtype=np.int32))
line_z.colors = o3d.utility.Vector3dVector(np.array([[0.15, 0.35, 1.0]], dtype=np.float64))

tick_half = 0.02
tick_z_vals = np.linspace(z_min, z_max, num=5)
tp, tl, tc = [], [], []
k = 0
for zt in tick_z_vals:
    tp.extend([[cx - tick_half, cy, zt], [cx + tick_half, cy, zt]])
    tl.append([k, k + 1])
    tc.append([0.2, 0.45, 1.0])
    k += 2
ticks_z = o3d.geometry.LineSet()
ticks_z.points = o3d.utility.Vector3dVector(np.asarray(tp, dtype=np.float64))
ticks_z.lines = o3d.utility.Vector2iVector(np.asarray(tl, dtype=np.int32))
ticks_z.colors = o3d.utility.Vector3dVector(np.asarray(tc, dtype=np.float64))

o3d.visualization.draw_geometries(
    [pcd, frame, line_z, ticks_z],
    window_name="Image 1 — Nuage (XY norm., Z = depth_1_masked brut)",
)


Img1 — Z brut (depth_map): min=0.40031, max=1, span=0.59969, std=0.122339, valeurs uniques (~8 déc.)=586663
Img1 — Z géométrique (affichage): min=0, max=0.5, span=0.5 (attendu ≈ 0.5 si le masque couvre min et max)


In [153]:
# Superposition 3D : XY normalisés, Z = valeurs brutes des depth_map masquées
# Teinte rougeâtre (image 1) vs bleutée (image 2)
import open3d as o3d

h_s, w_s = depth_1_masked.shape
if depth_2_masked.shape != (h_s, w_s):
    raise ValueError("depth_1_masked et depth_2_masked doivent avoir la meme taille.")

s_xy = float(max(h_s, w_s))


def _rgb_cropped_to_depth_grid(rgb_source, h, w):
    arr = np.array(rgb_source)
    if arr.dtype != np.uint8:
        arr = (np.clip(arr, 0, 1) * 255).astype(np.uint8)
    if arr.shape[0] != h or arr.shape[1] != w:
        arr = np.asarray(
            Image.fromarray(arr).resize((w, h), Image.Resampling.LANCZOS),
            dtype=np.uint8,
        )
    return arr


def _pcd_from_masked_depth(z_masked, rgb_uint8, tint_rgb, mix, h_img, w_img, s_xy_, x_offset=0.0):
    """Z = valeurs brutes du tableau de profondeur masqué (même échelle que la depth map)."""
    xx, yy = np.meshgrid(np.arange(w_img), np.arange(h_img), indexing="xy")
    x_n = xx.astype(np.float64) / s_xy_ + float(x_offset)
    y_n = yy.astype(np.float64) / s_xy_
    z_n = np.where(
        np.isfinite(z_masked),
        z_masked.astype(np.float64),
        np.nan,
    )
    pts = np.stack([x_n, y_n, z_n], axis=-1).reshape(-1, 3).astype(np.float64)
    base = rgb_uint8.reshape(-1, 3).astype(np.float64) / 255.0
    tint = np.asarray(tint_rgb, dtype=np.float64).reshape(1, 3)
    col = (1.0 - mix) * base + mix * tint
    col = np.clip(col, 0.0, 1.0)
    valid = np.isfinite(pts[:, 2])
    idx = np.where(valid)[0]
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts[idx])
    pcd.colors = o3d.utility.Vector3dVector(col[idx])
    return pcd


rgb1 = _rgb_cropped_to_depth_grid(image_cropped_1, h_s, w_s)
rgb2 = _rgb_cropped_to_depth_grid(image_cropped_2, h_s, w_s)

z1v = depth_1_masked[np.isfinite(depth_1_masked)]
z2v = depth_2_masked[np.isfinite(depth_2_masked)]
if z1v.size == 0 and z2v.size == 0:
    raise ValueError("Aucune profondeur valide pour les nuages.")

# Stats sur le masque (pour axe Z de visualisation et contrôle span)
if z1v.size:
    z1_min, z1_max = float(z1v.min()), float(z1v.max())
else:
    z1_min, z1_max = 0.0, 1.0
if z2v.size:
    z2_min, z2_max = float(z2v.min()), float(z2v.max())
else:
    z2_min, z2_max = 0.0, 1.0
for name, zv in [("depth_1_masked", z1v), ("depth_2_masked", z2v)]:
    if zv.size:
        zu = len(np.unique(np.round(zv.astype(np.float64), 8)))
        print(
            f"{name}: Z brut min={float(zv.min()):.6g}, max={float(zv.max()):.6g}, "
            f"std={float(np.std(zv)):.6g}, uniques≈{zu}"
        )
if z1v.size and max(z1_max - z1_min, 0.0) < 1e-9:
    raise ValueError("depth_1 : Z quasi constante sur le masque.")
if z2v.size and max(z2_max - z2_min, 0.0) < 1e-9:
    raise ValueError("depth_2 : Z quasi constante sur le masque.")

# Même position XY pour les deux nuages (superposition) ; Z = profondeur respective par image.
x_offset_1 = 0.0
x_offset_2 = 0.0

# Teintes (RGB 0-1) : rouge/orange vs cyan/bleu — mélange avec RGB pour différencier les deux images
tint_1 = np.array([0.92, 0.38, 0.28])
tint_2 = np.array([0.25, 0.55, 0.95])

pcd_super_1 = _pcd_from_masked_depth(
    depth_1_masked, rgb1, tint_1, 0.5, h_s, w_s, s_xy, x_offset=x_offset_1
)
pcd_super_2 = _pcd_from_masked_depth(
    depth_2_masked, rgb2, tint_2, 0.5, h_s, w_s, s_xy, x_offset=x_offset_2
)
z_vis_1 = np.asarray(pcd_super_1.points)[:, 2]
z_vis_2 = np.asarray(pcd_super_2.points)[:, 2]
print(
    f"Z (= depth_map) nuage1: [{z_vis_1.min():.6g}, {z_vis_1.max():.6g}] | "
    f"nuage2: [{z_vis_2.min():.6g}, {z_vis_2.max():.6g}]"
)

# Repère + axe Z sur la plage couverte par les deux nuages (profondeurs brutes)
cy = 0.5 * (h_s - 1) / s_xy
cx = 0.5 * (w_s - 1) / s_xy
if z1v.size and z2v.size:
    z_axis_lo = min(z1_min, z2_min)
    z_axis_hi = max(z1_max, z2_max)
elif z1v.size:
    z_axis_lo, z_axis_hi = z1_min, z1_max
else:
    z_axis_lo, z_axis_hi = z2_min, z2_max
z_axis_span = max(z_axis_hi - z_axis_lo, 1e-12)
axis_size = max(0.08, 0.15 * z_axis_span)
frame = o3d.geometry.TriangleMesh.create_coordinate_frame(
    size=axis_size, origin=[cx, cy, z_axis_lo]
)


def _z_axis_and_ticks(cx_, cy_, z_lo, z_hi):
    lz_pts = np.array([[cx_, cy_, z_lo], [cx_, cy_, z_hi]], dtype=np.float64)
    lz = o3d.geometry.LineSet()
    lz.points = o3d.utility.Vector3dVector(lz_pts)
    lz.lines = o3d.utility.Vector2iVector(np.array([[0, 1]], dtype=np.int32))
    lz.colors = o3d.utility.Vector3dVector(np.array([[0.15, 0.35, 1.0]], dtype=np.float64))

    tick_half = 0.02
    tp, tl, tc = [], [], []
    k = 0
    for zt in np.linspace(z_lo, z_hi, num=5):
        tp.extend([[cx_ - tick_half, cy_, zt], [cx_ + tick_half, cy_, zt]])
        tl.append([k, k + 1])
        tc.append([0.2, 0.45, 1.0])
        k += 2
    tz = o3d.geometry.LineSet()
    tz.points = o3d.utility.Vector3dVector(np.asarray(tp, dtype=np.float64))
    tz.lines = o3d.utility.Vector2iVector(np.asarray(tl, dtype=np.int32))
    tz.colors = o3d.utility.Vector3dVector(np.asarray(tc, dtype=np.float64))
    return lz, tz


line_z, ticks_z = _z_axis_and_ticks(cx, cy, z_axis_lo, z_axis_hi)

win = (
    f"Superposition — Z = depth brutes [{z_axis_lo:.4g}, {z_axis_hi:.4g}] ; "
    f"img1 rouge [{z1_min:.4g}, {z1_max:.4g}] ; img2 bleu [{z2_min:.4g}, {z2_max:.4g}]"
)
o3d.visualization.draw_geometries(
    [pcd_super_1, pcd_super_2, frame, line_z, ticks_z],
    window_name=win,
)

depth_1_masked: Z brut min=0.40031, max=1, std=0.122339, uniques≈586663
depth_2_masked: Z brut min=0.399506, max=0.998325, std=0.139041, uniques≈594583
Z géom. nuage1: [0, 1] | nuage2: [0, 1]


In [148]:
# Relief 3D Image 1 : XY / max(h,w), Z = depth_1_masked brut
rgb_img = np.asarray(image_cropped_1, dtype=np.uint8)
if rgb_img.ndim == 2:
    rgb_img = np.stack([rgb_img] * 3, axis=-1)
h, w = depth_1_masked.shape
if rgb_img.shape[0] != h or rgb_img.shape[1] != w:
    rgb_img = np.asarray(Image.fromarray(rgb_img).resize((w, h), Image.Resampling.LANCZOS), dtype=np.uint8)

s_xy = float(max(h, w))
cols = np.arange(w, dtype=np.float64)
rows = np.arange(h, dtype=np.float64)
points = np.empty((h, w, 3), dtype=np.float64)
points[:, :, 0] = (cols / s_xy)[np.newaxis, :]
points[:, :, 1] = (rows / s_xy)[:, np.newaxis]
z_geom = np.where(
    np.isfinite(depth_1_masked),
    depth_1_masked.astype(np.float64),
    np.nan,
)
points[:, :, 2] = z_geom
valid = np.isfinite(points[:, :, 2])
points_valid = points[valid]
colors_valid = (rgb_img.reshape(-1, 3) / 255.0).astype(np.float64)[valid.ravel()]
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points_valid)
pcd.colors = o3d.utility.Vector3dVector(colors_valid)
o3d.visualization.draw_geometries([pcd], window_name="Image 1 — Relief 3D (Z = depth brut)")


In [140]:
# Cartes de profondeur : comparaison et difference avec depth_2_aligned
h1, w1 = depth_1.shape
depth_2_aligned = np.asarray(depth_2_aligned, dtype=np.float64)
if depth_2_aligned.shape != (h1, w1):
    depth_2_aligned = cv2.resize(
        depth_2_aligned.astype(np.float32), (w1, h1), interpolation=cv2.INTER_LINEAR
    ).astype(np.float64)

depth_1_f = depth_1.astype(np.float64)
depth_2_f = depth_2_aligned
diff_map = depth_1_f - depth_2_f
diff_map_masked = np.where(mask_combined == 255, diff_map, np.nan)

fig, axes = plt.subplots(3, 3, figsize=(14, 12))
v_abs = np.nanmax(np.abs(diff_map)) if np.any(np.isfinite(diff_map)) else 1.0
if v_abs < 1e-12:
    v_abs = 1.0
v_max_d = max(np.nanmax(depth_1_f), np.nanmax(depth_2_f)) if np.any(np.isfinite(depth_1_f)) else 1.0

# Ligne 1 : RGB + difference brute
axes[0, 0].imshow(image_cropped_1)
axes[0, 0].set_title("Image 1 (33 g) — RGB")
axes[0, 0].axis("off")
axes[0, 1].imshow(image_cropped_2)
axes[0, 1].set_title("Image 2 (25 g) — RGB")
axes[0, 1].axis("off")
im0 = axes[0, 2].imshow(diff_map, cmap="coolwarm", vmin=-v_abs, vmax=v_abs)
axes[0, 2].set_title("Difference brute: depth_1 - depth_2_aligned")
axes[0, 2].axis("off")
plt.colorbar(im0, ax=axes[0, 2], fraction=0.046, label="Delta profondeur")

# Ligne 2 : profondeurs masquees + difference masquee
depth_1_display = np.where(mask_combined == 255, depth_1_f, np.nan)
depth_2_display = np.where(mask_combined == 255, depth_2_f, np.nan)
im_d1 = axes[1, 0].imshow(depth_1_display, cmap="plasma", vmin=0, vmax=v_max_d)
axes[1, 0].set_title("depth_1 (masque commun)")
axes[1, 0].axis("off")
plt.colorbar(im_d1, ax=axes[1, 0], fraction=0.046, label="Profondeur")
im_d2 = axes[1, 1].imshow(depth_2_display, cmap="plasma", vmin=0, vmax=v_max_d)
axes[1, 1].set_title("depth_2_aligned (masque commun)")
axes[1, 1].axis("off")
plt.colorbar(im_d2, ax=axes[1, 1], fraction=0.046, label="Profondeur")
im1 = axes[1, 2].imshow(diff_map_masked, cmap="coolwarm", vmin=-v_abs, vmax=v_abs)
axes[1, 2].set_title("Difference masquee")
axes[1, 2].axis("off")
plt.colorbar(im1, ax=axes[1, 2], fraction=0.046, label="Delta profondeur")

# Ligne 3 : masque commun
axes[2, 0].imshow(mask_combined, cmap="gray")
axes[2, 0].set_title("Masque commun (union)")
axes[2, 0].axis("off")
axes[2, 1].axis("off")
axes[2, 2].axis("off")

plt.tight_layout()
plt.show()

In [89]:
# Différence par pixel des deux cartes de profondeur (masquées), puis somme
diff_pixels = depth_1_masked - depth_2_masked
somme_diff = np.nansum(diff_pixels)
filtered_diff_pixels = diff_pixels[np.isfinite(diff_pixels)]
filtered_diff_pixels.sum()


np.float64(-117346.52115287907)

In [141]:
# Nuage de points : Z = depth_1 - depth_2_aligned (différence sur la carte alignée)
h1, w1 = depth_1.shape
depth_2_aligned = np.asarray(depth_2_aligned, dtype=np.float64)
if depth_2_aligned.shape != (h1, w1):
    depth_2_aligned = cv2.resize(
        depth_2_aligned.astype(np.float32), (w1, h1), interpolation=cv2.INTER_LINEAR
    ).astype(np.float64)
diff_z = depth_1.astype(np.float64) - depth_2_aligned
# On garde uniquement les pixels objet (mask_obj) pour éviter le bruit du fond
diff_masked = np.where(mask_combined == 255, diff_z, np.nan)

rgb_diff = np.array(image_cropped_1)
if rgb_diff.dtype != np.uint8:
    rgb_diff = (np.clip(rgb_diff, 0, 1) * 255).astype(np.uint8)
if rgb_diff.shape[0] != h1 or rgb_diff.shape[1] != w1:
    rgb_diff = np.asarray(Image.fromarray(rgb_diff).resize((w1, h1), Image.Resampling.LANCZOS), dtype=np.uint8)

xx_d, yy_d = np.meshgrid(np.arange(w1), np.arange(h1))
pts_diff = np.stack([xx_d, yy_d, diff_masked], axis=-1).reshape(-1, 3)
colors_diff = rgb_diff.reshape(-1, 3) / 255.0
pcd_diff = o3d.geometry.PointCloud()
pcd_diff.points = o3d.utility.Vector3dVector(pts_diff)
pcd_diff.colors = o3d.utility.Vector3dVector(colors_diff)
valid_d = np.isfinite(pts_diff[:, 2])
pcd_diff = pcd_diff.select_by_index(np.where(valid_d)[0])

# Option : coordonnées spatiales pour un relief lisible (Z = différence, valeurs brutes)
scale_xy_d = 1.0 / w1
pts_sp = np.column_stack([
    pts_diff[valid_d, 0] * scale_xy_d,
    pts_diff[valid_d, 1] * scale_xy_d,
    pts_diff[valid_d, 2],
])
pcd_diff.points = o3d.utility.Vector3dVector(pts_sp)

o3d.visualization.draw_geometries(
    [pcd_diff],
    window_name="Nuage de points : Z = Image 1 − Image 2 (depth_2_aligned)"
)

### Géométrisation — Différence (Z = Image 1 − Image 2)

Le nuage `pcd_diff` utilise **depth_1 − depth_2_aligned** (carte 2 alignée sur la grille de l’image 1). Même pipeline que pour l’image 1 : downsampling, enrichissement par points en Z=0, puis reconstruction de surface (mesh + normales).

In [91]:
# Downsample du nuage différence (pcd_diff) pour alléger la géométrisation
voxel_size_diff = 0.002
pcd_diff_ds = pcd_diff.voxel_down_sample(voxel_size_diff)
o3d.visualization.draw_geometries(
    [pcd_diff_ds],
    window_name="Différence (Image 1 − Image 2) — Relief 3D (downsampled)"
)

In [138]:
# Reconstruction de surface par Poisson sur le nuage différence (comme Image 1)

# (On n'utilise plus pyvista ici, mais le Poisson surface reconstruction d'Open3D)
# Attention : nécessite des normales cohérentes sur le nuage

# Nettoyage et recalcul des normales
pcd_diff_pv = pcd_diff_ds.voxel_down_sample(0.005)
pcd_diff_pv.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.02, max_nn=30),
    fast_normal_computation=True
)
# Oriente les normales de façon cohérente
pcd_diff_pv.orient_normals_consistent_tangent_plane(k=10)

# Reconstruction par Poisson
mesh_diff, densities_diff = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
    pcd_diff_pv, depth=8
)

# Optionnel : suppression des triangles de faible densité (pour nettoyer)
densities_diff = np.asarray(densities_diff)
density_threshold = np.quantile(densities_diff, 0.02)
vertices_to_remove = densities_diff < density_threshold
mesh_diff.remove_vertices_by_mask(vertices_to_remove)
mesh_diff.compute_vertex_normals()
normals_diff = np.asarray(mesh_diff.vertex_normals)
verts_diff = np.asarray(mesh_diff.vertices)

# Lignes des normales (indices corrects)
step_n = max(len(normals_diff) // 300, 1)
line_pts_diff = []
for i in range(0, len(normals_diff), step_n):
    start = verts_diff[i]
    end = start + normals_diff[i] * 0.02
    line_pts_diff.append(start)
    line_pts_diff.append(end)
n_ln = len(line_pts_diff) // 2
line_idx_diff = np.array([[2*k, 2*k+1] for k in range(n_ln)], dtype=np.int32)
line_set_diff = o3d.geometry.LineSet()
line_set_diff.points = o3d.utility.Vector3dVector(np.asarray(line_pts_diff))
line_set_diff.lines = o3d.utility.Vector2iVector(line_idx_diff)
line_set_diff.colors = o3d.utility.Vector3dVector(np.tile([1, 0, 0], (n_ln, 1)))

# Arêtes de bord : arêtes qui n'appartiennent qu'à un seul triangle
tris_diff = np.asarray(mesh_diff.triangles)
edge_count = {}
for (a, b, c) in tris_diff:
    for u, v in [(a, b), (b, c), (c, a)]:
        e = (min(u, v), max(u, v))
        edge_count[e] = edge_count.get(e, 0) + 1
boundary_edges = np.array([e for e, c in edge_count.items() if c == 1], dtype=np.int32)
line_set_boundary = o3d.geometry.LineSet()
line_set_boundary.points = mesh_diff.vertices
line_set_boundary.lines = o3d.utility.Vector2iVector(boundary_edges)
line_set_boundary.paint_uniform_color([0, 1, 0])  # vert = bord du mesh

geoms_diff = [mesh_diff]
if n_ln > 0:
    geoms_diff.append(line_set_diff)
geoms_diff.append(line_set_boundary)
o3d.visualization.draw_geometries(
    geoms_diff,
    window_name="Différence (Image 1 − Image 2) — Mesh Poisson + normales + bord"
)

In [93]:
geoms_diff[0].is_edge_manifold(),geoms_diff[0].is_vertex_manifold()

(False, True)

### Visualisations 3D — Image 2 (25 g)

Nuage de points et relief 3D pour la **deuxième image** : Z = `depth_2_masked`, construit à partir de **`depth_2_aligned`** dans le masque commun (`mask_combined`), couleurs depuis `image_cropped_2`. Permet de comparer les deux prises (33 g vs 25 g) sur la même échelle de profondeur que l’image 1.

In [142]:
# Nuage de points Image 2 : XY normalisés, Z = depth_2_masked brut (même échelle que la depth map)
# + teinte bleue pour cohérence avec la superposition deux images
rgb_img_2 = np.array(image_cropped_2)
if rgb_img_2.dtype != np.uint8:
    rgb_img_2 = (np.clip(rgb_img_2, 0, 1) * 255).astype(np.uint8)
h2, w2 = depth_2_masked.shape
if rgb_img_2.shape[0] != h2 or rgb_img_2.shape[1] != w2:
    rgb_img_2 = np.asarray(
        Image.fromarray(rgb_img_2).resize((w2, h2), Image.Resampling.LANCZOS),
        dtype=np.uint8,
    )

xx_2, yy_2 = np.meshgrid(np.arange(w2), np.arange(h2), indexing="xy")
s_xy_2 = float(max(w2, h2))
x_n = xx_2.astype(np.float64) / s_xy_2
y_n = yy_2.astype(np.float64) / s_xy_2

zv2 = depth_2_masked[np.isfinite(depth_2_masked)]
if zv2.size == 0:
    raise ValueError("depth_2_masked : aucun pixel valide.")
z2_lo, z2_hi = float(zv2.min()), float(zv2.max())
z_span2 = max(z2_hi - z2_lo, 1e-12)
z_n = np.where(
    np.isfinite(depth_2_masked),
    depth_2_masked.astype(np.float64),
    np.nan,
)
points_2 = np.stack([x_n, y_n, z_n], axis=-1).reshape(-1, 3)

tint_2 = np.array([0.25, 0.55, 0.95])
base_2 = rgb_img_2.reshape(-1, 3).astype(np.float64) / 255.0
colors_2 = np.clip(0.5 * base_2 + 0.5 * tint_2, 0.0, 1.0)

pcd_2 = o3d.geometry.PointCloud()
pcd_2.points = o3d.utility.Vector3dVector(points_2)
pcd_2.colors = o3d.utility.Vector3dVector(colors_2)
valid_2 = np.isfinite(points_2[:, 2])
pcd_2 = pcd_2.select_by_index(np.where(valid_2)[0])

cx2, cy2 = 0.5 * (w2 - 1) / s_xy_2, 0.5 * (h2 - 1) / s_xy_2
axis_size = max(0.08, 0.15 * z_span2)
frame_2 = o3d.geometry.TriangleMesh.create_coordinate_frame(size=axis_size, origin=[cx2, cy2, z2_lo])
line_z_pts = np.array([[cx2, cy2, z2_lo], [cx2, cy2, z2_hi]], dtype=np.float64)
line_z_2 = o3d.geometry.LineSet()
line_z_2.points = o3d.utility.Vector3dVector(line_z_pts)
line_z_2.lines = o3d.utility.Vector2iVector(np.array([[0, 1]], dtype=np.int32))
line_z_2.colors = o3d.utility.Vector3dVector(np.array([[0.15, 0.35, 1.0]], dtype=np.float64))
tick_half = 0.02
tp, tl, tc = [], [], []
k = 0
for zt in np.linspace(z2_lo, z2_hi, num=5):
    tp.extend([[cx2 - tick_half, cy2, zt], [cx2 + tick_half, cy2, zt]])
    tl.append([k, k + 1])
    tc.append([0.2, 0.45, 1.0])
    k += 2
ticks_z_2 = o3d.geometry.LineSet()
ticks_z_2.points = o3d.utility.Vector3dVector(np.asarray(tp, dtype=np.float64))
ticks_z_2.lines = o3d.utility.Vector2iVector(np.asarray(tl, dtype=np.int32))
ticks_z_2.colors = o3d.utility.Vector3dVector(np.asarray(tc, dtype=np.float64))

o3d.visualization.draw_geometries(
    [pcd_2, frame_2, line_z_2, ticks_z_2],
    window_name="Image 2 — Nuage (XY norm., Z = depth_2 brut, teinte bleue)",
)

In [162]:
# Relief 3D Image 2 : Z = depth_2_masked brut (même échelle que la depth map)
scale_xy_2 = 1.0 / w2
cols_2 = np.arange(w2, dtype=np.float64)
rows_2 = np.arange(h2, dtype=np.float64)
points_spatial_2 = np.empty((h2, w2, 3), dtype=np.float64)
points_spatial_2[:, :, 0] = (cols_2 * scale_xy_2)[np.newaxis, :]
points_spatial_2[:, :, 1] = (rows_2 * scale_xy_2)[:, np.newaxis]
points_spatial_2[:, :, 2] = depth_2_masked.astype(np.float64)
valid_sp_2 = np.isfinite(points_spatial_2[:, :, 2]) & (points_spatial_2[:, :, 2] > -1)
points_valid_2 = points_spatial_2[valid_sp_2]
colors_valid_2 = (rgb_img_2.reshape(-1, 3) / 255.0).astype(np.float64)[valid_sp_2.ravel()]
pcd_2_spatial = o3d.geometry.PointCloud()
pcd_2_spatial.points = o3d.utility.Vector3dVector(points_valid_2)
pcd_2_spatial.colors = o3d.utility.Vector3dVector(colors_valid_2)
o3d.visualization.draw_geometries(
    [pcd_2_spatial],
    window_name="Image 2 — Relief 3D (depth_2_aligned + masque commun)"
)

In [97]:
# Réaffichage relief Image 1 (pcd_z_pos n'existe plus sans soustraction)
o3d.visualization.draw_geometries([pcd], window_name="Image 1 — Relief 3D")


In [98]:
# Downsample du nuage Image 1 pour alléger la visualisation
voxel_size = 0.002
pcd_1_ds = pcd.voxel_down_sample(voxel_size)
o3d.visualization.draw_geometries([pcd_1_ds], window_name="Image 1 — Relief 3D (downsampled)")


In [144]:
# Nuage Image 1 enrichi : points du relief + projections sur Z=0 (à partir de pcd_1_ds)
points_1 = np.asarray(pcd_1_ds.points)
colors_1 = np.asarray(pcd_1_ds.colors)
extra_pts = np.column_stack([points_1[:, 0], points_1[:, 1], np.zeros(len(points_1))])
all_pts = np.vstack([points_1, extra_pts])
all_col = np.vstack([colors_1, colors_1])
pcd_augmented_pos = o3d.geometry.PointCloud()
pcd_augmented_pos.points = o3d.utility.Vector3dVector(all_pts)
pcd_augmented_pos.colors = o3d.utility.Vector3dVector(all_col)
o3d.visualization.draw_geometries([pcd_1_ds], window_name="Image 1 — Relief + projections Z=0")


In [145]:
# Volume (m³) par voxelisation à partir du nuage pcd_augmented_pos
# Échelle inventée : la plus grande dimension du nuage = 1 m

pts = np.asarray(pcd_1_ds.points)
min_pt = pts.min(axis=0)
max_pt = pts.max(axis=0)
extent = max_pt - min_pt
max_extent = extent.max()
# Mise à l'échelle : 1 m pour la plus grande dimension
scale_to_m = 1.0 / max_extent if max_extent > 0 else 1.0
pts_m = pts * scale_to_m

pcd_m = o3d.geometry.PointCloud()
pcd_m.points = o3d.utility.Vector3dVector(pts_m)
if pcd_1_ds.has_colors():
    pcd_m.colors = pcd_1_ds.colors

# Voxelisation : taille de voxel en m (ex. 2 cm)
voxel_size_m = 0.01
voxel_grid = o3d.geometry.VoxelGrid.create_from_point_cloud(pcd_1_ds, voxel_size=voxel_size_m)
n_voxels = len(voxel_grid.get_voxels())
volume_m3 = n_voxels * (voxel_size_m ** 3)

print(f"Échelle : {max_extent:.4f} unités → 1 m")
print(f"Voxel size : {voxel_size_m*100:.1f} cm")
print(f"Nombre de voxels occupés : {n_voxels}")
print(f"Volume (voxelisation) : {volume_m3:.4f} m³")

Échelle : 0.9091 unités → 1 m
Voxel size : 1.0 cm
Nombre de voxels occupés : 9091
Volume (voxelisation) : 0.0091 m³


In [101]:
o3d.visualization.draw_geometries(
    [voxel_grid],
    window_name="PyVista reconstruct_surface (mesh)",
)

In [102]:
np.linalg.norm(np.array((0.25,0.50,0.8))-np.array((0.75,0.75,0.4)))

np.float64(0.687386354243376)

In [103]:
def mesh_poisson(pcd, depth=8, density_quantile=0.03):
    """Reconstruction Poisson : surface lisse et fermée, ~20x plus rapide que ball pivoting."""
    if not pcd.has_normals():
        pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.05, max_nn=30))
    mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=depth)
    densities = np.asarray(densities)
    mesh = mesh.select_by_index(np.where(densities > np.quantile(densities, density_quantile))[0])
    mesh.remove_degenerate_triangles()
    mesh.remove_duplicated_triangles()
    mesh.compute_vertex_normals()
    return mesh

In [4]:
# Affiche le nuage de points dans un viewer Open3D
ply = r"C:\Users\mvm\pointcloud_pipeline\output\cloud_downsampled.ply"
pcd = o3d.io.read_point_cloud(ply)
o3d.visualization.draw_geometries([pcd], window_name="Nuage de points (Open3D)")

In [105]:
import cv2 as cv
# Top-hat : soustraire le fond (ouverture) pour faire ressortir les petits détails clairs (ex. sel sur page blanche)
# Puis Otsu sur l'image top-hat donne un seuillage plus fiable que Otsu direct sur l'originale.
img_th = cv.imread(r"C:\Users\mvm\open3d_vision\data\photos sel\33 grammes.jpg", cv.IMREAD_GRAYSCALE)
assert img_th is not None, "Image non trouvée"

# Élément structurant : taille à adapter à la taille des objets à détecter (plus gros = moins de bruit, grains plus gros gardés)
equalized = cv2.equalizeHist(img_th)
kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (25, 25))

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
clahe_equalized = clahe.apply(img_th)

tophat = cv.morphologyEx(equalized, cv.MORPH_TOPHAT, kernel)

# Visualisation : image originale, equalized, CLAHE, top-hat seul, puis top-hat + Otsu
_, th_tophat_otsu = cv.threshold(clahe_equalized, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
axes[0].imshow(img_th, cmap="gray")
axes[0].set_title("Original (niveau de gris)")
axes[0].axis("off")
axes[1].imshow(equalized, cmap="gray")
axes[1].set_title("Equalized")
axes[1].axis("off")
axes[2].imshow(clahe_equalized, cmap="gray")
axes[2].set_title("CLAHE")
axes[2].axis("off")
axes[3].imshow(tophat, cmap="gray")
axes[3].set_title("Top-hat (détails clairs)")
axes[3].axis("off")
axes[4].imshow(th_tophat_otsu, cmap="gray")
axes[4].set_title("Otsu")
axes[4].axis("off")
plt.tight_layout()
plt.show()